<a href="https://colab.research.google.com/github/F1ameX/Modern-Methods-of-Deep-Machine-Learning/blob/main/5_residual_neural_network/5_residual_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset

from torchvision import datasets, transforms

from tqdm.auto import tqdm

In [ ]:
SEED = 52
BATCH_SIZE = 128

FAST_EPOCHS = 3
FINAL_EPOCHS = 15

TRAIN_SIZE = 50_000
VAL_SIZE = 10_000

MINI_TRAIN_PER_CLASS = 300
MINI_VAL_PER_CLASS   = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE: cuda


In [ ]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        *,
        variant: str = "a",
        conv_kernel_size: int = 3,
        conv_stride: int = 1,
        conv_padding: int | None = None,
        use_batchnorm: bool = True,
        pool_kernel_size: int = 2,
        pool_stride: int = 2,
        dropout: float = 0.0,
    ):
        super().__init__()
        assert variant in ("a", "b")

        if conv_padding is None:
            conv_padding = conv_kernel_size // 2

        layers = []

        layers.append(
            nn.Conv2d(
                input_channels,
                output_channels,
                kernel_size=conv_kernel_size,
                stride=conv_stride,
                padding=conv_padding,
                bias=not use_batchnorm,
            )
        )
        if use_batchnorm:
            layers.append(nn.BatchNorm2d(output_channels))
        layers.append(nn.ReLU(inplace=True))

        if variant == "b":
            layers.append(
                nn.Conv2d(
                    output_channels,
                    output_channels,
                    kernel_size=conv_kernel_size,
                    stride=1,
                    padding=conv_padding,
                    bias=not use_batchnorm,
                )
            )
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(output_channels))
            layers.append(nn.ReLU(inplace=True))

        layers.append(nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride))

        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ResidualBlock(nn.Module):
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        *,
        conv_kernel_size: int = 3,
        conv_stride: int = 1,
        conv_padding: int | None = None,
        use_batchnorm: bool = True,
        dropout: float = 0.0,
    ):
        super().__init__()

        if conv_padding is None:
            conv_padding = conv_kernel_size // 2

        self.conv1 = nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size=conv_kernel_size,
            stride=conv_stride,
            padding=conv_padding,
            bias=not use_batchnorm,
        )
        self.bn1 = nn.BatchNorm2d(output_channels) if use_batchnorm else nn.Identity()

        self.conv2 = nn.Conv2d(
            output_channels,
            output_channels,
            kernel_size=conv_kernel_size,
            stride=1,
            padding=conv_padding,
            bias=not use_batchnorm,
        )
        self.bn2 = nn.BatchNorm2d(output_channels) if use_batchnorm else nn.Identity()

        if (input_channels != output_channels) or (conv_stride != 1):
            self.skip = nn.Sequential(
                nn.Conv2d(input_channels, output_channels, kernel_size=1, stride=conv_stride, bias=False),
                nn.BatchNorm2d(output_channels) if use_batchnorm else nn.Identity(),
            )
        else:
            self.skip = nn.Identity()

        self.activation = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.activation(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + self.skip(x)
        out = self.activation(out)
        out = self.dropout(out)
        return out


class BottleneckBlock(nn.Module):
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        *,
        bottleneck_channels: int | None = None,
        conv_kernel_size: int = 3,
        conv_stride: int = 1,
        conv_padding: int | None = None,
        use_batchnorm: bool = True,
        dropout: float = 0.0,
    ):
        super().__init__()

        if conv_padding is None:
            conv_padding = conv_kernel_size // 2

        if bottleneck_channels is None:
            bottleneck_channels = max(8, output_channels // 4)

        self.conv1 = nn.Conv2d(input_channels, bottleneck_channels, kernel_size=1, stride=1, padding=0, bias=not use_batchnorm)
        self.bn1 = nn.BatchNorm2d(bottleneck_channels) if use_batchnorm else nn.Identity()

        self.conv2 = nn.Conv2d(
            bottleneck_channels,
            bottleneck_channels,
            kernel_size=conv_kernel_size,
            stride=conv_stride,
            padding=conv_padding,
            bias=not use_batchnorm,
        )
        self.bn2 = nn.BatchNorm2d(bottleneck_channels) if use_batchnorm else nn.Identity()

        self.conv3 = nn.Conv2d(bottleneck_channels, output_channels, kernel_size=1, stride=1, padding=0, bias=not use_batchnorm)
        self.bn3 = nn.BatchNorm2d(output_channels) if use_batchnorm else nn.Identity()

        if (input_channels != output_channels) or (conv_stride != 1):
            self.skip = nn.Sequential(
                nn.Conv2d(input_channels, output_channels, kernel_size=1, stride=conv_stride, bias=False),
                nn.BatchNorm2d(output_channels) if use_batchnorm else nn.Identity(),
            )
        else:
            self.skip = nn.Identity()

        self.activation = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.activation(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.activation(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out = out + self.skip(x)
        out = self.activation(out)
        out = self.dropout(out)
        return out


class ResNetLikeCNN(nn.Module):
    def __init__(
        self,
        *,
        input_channels: int = 3,
        architecture: str = "res",
        conv_variant: str = "a",
        n_blocks: int = 3,
        base_channels: int = 32,
        channels_multiplier: int = 2,
        use_batchnorm: bool = True,
        dropout: float = 0.0,
        num_classes: int = 10,

        conv_kernel_size: int = 3,
        conv_stride: int = 1,
        conv_padding: int | None = None,

        pool_kernel_size: int = 2,
        pool_stride: int = 2,
    ):
        super().__init__()
        assert architecture in ("conv", "res", "bottle")
        assert conv_variant in ("a", "b")
        assert n_blocks >= 1

        layers = []
        channels = base_channels

        layers.extend([
            nn.Conv2d(input_channels, channels, kernel_size=3, stride=1, padding=1, bias=not use_batchnorm),
            nn.BatchNorm2d(channels) if use_batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ])

        input_channels = channels

        for block_id in range(n_blocks):
            stride_here = 2 if (architecture in ("res", "bottle") and block_id > 0 and block_id % 2 == 0) else 1

            if architecture == "conv":
                layers.append(
                    ConvBlock(
                        input_channels=input_channels,
                        output_channels=channels,
                        variant=conv_variant,
                        conv_kernel_size=conv_kernel_size,
                        conv_stride=conv_stride,
                        conv_padding=conv_padding,
                        use_batchnorm=use_batchnorm,
                        pool_kernel_size=pool_kernel_size,
                        pool_stride=pool_stride,
                        dropout=dropout,
                    )
                )
                input_channels = channels
                channels *= channels_multiplier

            elif architecture == "res":
                layers.append(
                    ResidualBlock(
                        input_channels=input_channels,
                        output_channels=channels,
                        conv_kernel_size=conv_kernel_size,
                        conv_stride=stride_here,
                        conv_padding=conv_padding,
                        use_batchnorm=use_batchnorm,
                        dropout=dropout,
                    )
                )
                input_channels = channels
                if stride_here == 2:
                    channels *= channels_multiplier

            else:  # "bottle"
                layers.append(
                    BottleneckBlock(
                        input_channels=input_channels,
                        output_channels=channels,
                        conv_kernel_size=conv_kernel_size,
                        conv_stride=stride_here,
                        conv_padding=conv_padding,
                        use_batchnorm=use_batchnorm,
                        dropout=dropout,
                    )
                )
                input_channels = channels
                if stride_here == 2:
                    channels *= channels_multiplier

        self.features = nn.Sequential(*layers)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(input_channels, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.gap(x).flatten(1)
        logits = self.classifier(x)
        return logits

In [ ]:
SEED = 52
BATCH_SIZE = 128
NUM_WORKERS = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616),
    ),
])

full_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

train_size = 45_000
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("CIFAR-10 splits:")
print("train:", len(train_dataset), "val:", len(val_dataset), "test:", len(test_dataset))

x0, y0 = next(iter(train_loader))
print("train batch:", x0.shape, y0.shape, "| labels:", y0[:10].tolist())

100%|██████████| 170M/170M [00:03<00:00, 48.7MB/s]


CIFAR-10 splits:
train: 45000 val: 5000 test: 10000
train batch: torch.Size([128, 3, 32, 32]) torch.Size([128]) | labels: [1, 4, 7, 9, 0, 4, 7, 2, 7, 0]


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

    return total_loss / total


@torch.no_grad()
def eval_loss_acc(model, loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels, reduction="sum")
        total_loss += float(loss.item())

        preds = logits.argmax(dim=1)
        correct += int((preds == labels).sum().item())
        total += int(labels.numel())

    return {"loss": total_loss / total, "acc": correct / total}


def fit_model(model, train_loader, val_loader, *, device, learning_rate, weight_decay, epochs, target_acc=0.90):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    first_epoch_reach = None
    history = []

    for ep in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = eval_loss_acc(model, val_loader, device)

        history.append((ep, train_loss, val_metrics["loss"], val_metrics["acc"]))

        if first_epoch_reach is None and val_metrics["acc"] >= target_acc:
            first_epoch_reach = ep

        print(
            f"epoch={ep:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['acc']:.4f}"
        )

    return model, history, first_epoch_reach


device = DEVICE

CONFIGS = [
    dict(name="conv_a_small",   architecture="conv",   conv_variant="a", n_blocks=3, base_channels=32, channels_multiplier=2,
         use_batchnorm=True, dropout=0.1, conv_kernel_size=3, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=1e-3, weight_decay=1e-4, epochs=15),

    dict(name="conv_b_small",   architecture="conv",   conv_variant="b", n_blocks=3, base_channels=32, channels_multiplier=2,
         use_batchnorm=True, dropout=0.1, conv_kernel_size=3, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=1e-3, weight_decay=1e-4, epochs=15),

    dict(name="conv_a_k5",      architecture="conv",   conv_variant="a", n_blocks=3, base_channels=32, channels_multiplier=2,
         use_batchnorm=True, dropout=0.1, conv_kernel_size=5, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=8e-4, weight_decay=1e-4, epochs=15),

    dict(name="res_3_blocks",   architecture="res",    conv_variant="a", n_blocks=3, base_channels=32, channels_multiplier=2,
         use_batchnorm=True, dropout=0.1, conv_kernel_size=3, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=8e-4, weight_decay=1e-4, epochs=15),

    dict(name="res_5_blocks",   architecture="res",    conv_variant="a", n_blocks=5, base_channels=32, channels_multiplier=2,
         use_batchnorm=True, dropout=0.15, conv_kernel_size=3, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=6e-4, weight_decay=1e-4, epochs=15),

    dict(name="bottle_3_blocks",architecture="bottle", conv_variant="a", n_blocks=3, base_channels=32, channels_multiplier=2,
         use_batchnorm=True, dropout=0.1, conv_kernel_size=3, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=8e-4, weight_decay=1e-4, epochs=15),

    dict(name="bottle_5_blocks",architecture="bottle", conv_variant="a", n_blocks=5, base_channels=32, channels_multiplier=2,
         use_batchnorm=True, dropout=0.15, conv_kernel_size=3, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=6e-4, weight_decay=1e-4, epochs=15),

    dict(name="res_wide",       architecture="res",    conv_variant="a", n_blocks=3, base_channels=64, channels_multiplier=2,
         use_batchnorm=True, dropout=0.2, conv_kernel_size=3, conv_stride=1, conv_padding=None, pool_kernel_size=2, pool_stride=2,
         learning_rate=6e-4, weight_decay=1e-4, epochs=10),
]

results = []

for cfg in CONFIGS:
    print("\n" + "=" * 22, cfg["name"], "=" * 22)

    model = ResNetLikeCNN(
        input_channels=3,
        architecture=cfg["architecture"],
        conv_variant=cfg["conv_variant"],
        n_blocks=cfg["n_blocks"],
        base_channels=cfg["base_channels"],
        channels_multiplier=cfg["channels_multiplier"],
        use_batchnorm=cfg["use_batchnorm"],
        dropout=cfg["dropout"],
        num_classes=10,
        conv_kernel_size=cfg["conv_kernel_size"],
        conv_stride=cfg["conv_stride"],
        conv_padding=cfg["conv_padding"],
        pool_kernel_size=cfg["pool_kernel_size"],
        pool_stride=cfg["pool_stride"],
    )

    t0 = time.time()
    model, history, first_ep = fit_model(
        model,
        train_loader,
        val_loader,
        device=device,
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        epochs=cfg["epochs"],
        target_acc=0.90,
    )
    dt = time.time() - t0

    val_metrics = {"loss": history[-1][2], "acc": history[-1][3]}
    test_metrics = eval_loss_acc(model, test_loader, device)

    results.append({
        "name": cfg["name"],
        "arch": cfg["architecture"],
        "variant": cfg["conv_variant"] if cfg["architecture"] == "conv" else "-",
        "n_blocks": cfg["n_blocks"],
        "base_ch": cfg["base_channels"],
        "k": cfg["conv_kernel_size"],
        "conv_s": cfg["conv_stride"],
        "pool_k": cfg["pool_kernel_size"],
        "pool_s": cfg["pool_stride"],
        "bn": cfg["use_batchnorm"],
        "dropout": cfg["dropout"],
        "lr": cfg["learning_rate"],
        "wd": cfg["weight_decay"],
        "epochs": cfg["epochs"],
        "first_ep_acc>=0.90": first_ep,
        "val_acc": val_metrics["acc"],
        "test_acc": test_metrics["acc"],
        "test_loss": test_metrics["loss"],
        "time_sec": dt,
    })

summary_df = pd.DataFrame(results).sort_values(by=["test_acc", "val_acc"], ascending=False).reset_index(drop=True)
display(summary_df)


====================== conv_a_small ======================
epoch=01 | train_loss=1.6048 | val_loss=1.3193 | val_acc=0.5408
epoch=02 | train_loss=1.3104 | val_loss=1.1465 | val_acc=0.6024
epoch=03 | train_loss=1.1862 | val_loss=1.0922 | val_acc=0.6122
epoch=04 | train_loss=1.1053 | val_loss=0.9671 | val_acc=0.6652
epoch=05 | train_loss=1.0330 | val_loss=0.9587 | val_acc=0.6530
epoch=06 | train_loss=0.9803 | val_loss=0.8886 | val_acc=0.6924
epoch=07 | train_loss=0.9385 | val_loss=0.8439 | val_acc=0.7054
epoch=08 | train_loss=0.8983 | val_loss=0.7970 | val_acc=0.7280
epoch=09 | train_loss=0.8680 | val_loss=0.7673 | val_acc=0.7378
epoch=10 | train_loss=0.8426 | val_loss=0.8091 | val_acc=0.7234
epoch=11 | train_loss=0.8165 | val_loss=0.7796 | val_acc=0.7334
epoch=12 | train_loss=0.7932 | val_loss=0.7210 | val_acc=0.7510
epoch=13 | train_loss=0.7708 | val_loss=0.6960 | val_acc=0.7656
epoch=14 | train_loss=0.7582 | val_loss=0.7005 | val_acc=0.7640
epoch=15 | train_loss=0.7351 | val_loss=0.68

,name,arch,variant,n_blocks,base_ch,k,conv_s,pool_k,pool_s,bn,dropout,lr,wd,epochs,first_ep_acc>=0.90,val_acc,test_acc,test_loss,time_sec
0,conv_a_k5,conv,a,3,32,5,1,2,2,True,0.10,0.0008,0.0001,15,None,0.8156,0.8164,0.539777,194.286217
1,conv_b_small,conv,b,3,32,3,1,2,2,True,0.10,0.0010,0.0001,15,None,0.8140,0.8133,0.566307,211.466082
2,conv_a_small,conv,a,3,32,3,1,2,2,True,0.10,0.0010,0.0001,15,None,0.7684,0.7641,0.686533,192.591618
3,res_5_blocks,res,-,5,32,3,1,2,2,True,0.15,0.0006,0.0001,15,None,0.7368,0.7376,0.732361,286.819165
4,res_3_blocks,res,-,3,32,3,1,2,2,True,0.10,0.0008,0.0001,15,None,0.7064,0.7086,0.830038,248.537976
5,res_wide,res,-,3,64,3,1,2,2,True,0.20,0.0006,0.0001,10,None,0.6974,0.6915,0.865606,209.545578
6,bottle_5_blocks,bottle,-,5,32,3,1,2,2,True,0.15,0.0006,0.0001,15,None,0.5522,0.5609,1.208602,230.538471
7,bottle_3_blocks,bottle,-,3,32,3,1,2,2,True,0.10,0.0008,0.0001,15,None,0.5208,0.5318,1.301748,210.637655
